In [0]:
# Databricks notebook: 01_load_raw_data

from pyspark.sql import functions as F
from pyspark.sql.types import StringType

BASE_PATH = "/Volumes/hackathon/default/railway_data"

SCHEDULES_JSON_PATH = f"{BASE_PATH}/raw/schedules.json"
ROUTES_CSV_PATH = f"{BASE_PATH}/raw/Schedule1.0.csv"
PRICE_CSV_PATH = f"{BASE_PATH}/raw/price_data.csv"

CATALOG = "hackathon"
SCHEMA = "default"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"USE {CATALOG}.{SCHEMA}")

In [0]:
# Load schedules.json
# Using the accessible volume path
ACCESSIBLE_PATH = "/Volumes/workspace/default/databricks_project"

raw_schedules = (
    spark.read
    .option("multiLine", "true")
    .json(f"{ACCESSIBLE_PATH}/schedules.json")
)

display(raw_schedules.limit(10))
raw_schedules.printSchema()

In [0]:
# Load Schedule1.0.csv
ACCESSIBLE_PATH = "/Volumes/workspace/default/databricks_project"

raw_routes = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{ACCESSIBLE_PATH}/Schedule1.0.csv")
)

display(raw_routes.limit(10))
raw_routes.printSchema()

In [0]:
# Load price_data.csv

raw_prices = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{ACCESSIBLE_PATH}/price_data.csv")
)

display(raw_prices.limit(10))
raw_prices.printSchema()


In [0]:
# Normalize schedules table

schedules = (
    raw_schedules
    .select(
        F.col("train_number").cast("string").alias("train_number"),
        F.col("train_name").cast("string").alias("train_name"),
        F.upper(F.trim(F.col("station_code").cast("string"))).alias("station_code"),
        F.trim(F.col("station_name").cast("string")).alias("station_name"),
        F.col("arrival").cast("string").alias("arrival"),
        F.col("departure").cast("string").alias("departure"),
        F.col("day").cast("int").alias("day"),
        F.col("id").cast("string").alias("stop_id")
    )
    .dropDuplicates()
)

display(schedules.limit(10))


In [0]:
# Normalize route table

routes = (
    raw_routes
    .select(
        F.col("trainNumber").cast("string").alias("train_number"),
        F.col("trainName").cast("string").alias("train_name"),
        F.upper(F.trim(F.col("stationFrom").cast("string"))).alias("route_from"),
        F.upper(F.trim(F.col("stationTo").cast("string"))).alias("route_to"),
        F.col("trainRunsOnMon").cast("string").alias("runs_on_mon"),
        F.col("trainRunsOnTue").cast("string").alias("runs_on_tue"),
        F.col("trainRunsOnWed").cast("string").alias("runs_on_wed"),
        F.col("trainRunsOnThu").cast("string").alias("runs_on_thu"),
        F.col("trainRunsOnFri").cast("string").alias("runs_on_fri"),
        F.col("trainRunsOnSat").cast("string").alias("runs_on_sat"),
        F.col("trainRunsOnSun").cast("string").alias("runs_on_sun"),
        F.col("timeStamp").cast("string").alias("timestamp"),
        F.col("stationList").cast("string").alias("station_list")
    )
    .dropDuplicates()
)

display(routes.limit(10))


In [0]:
# Normalize price table

prices = (
    raw_prices
    .select(
        F.col("trainNumber").cast("string").alias("train_number"),
        F.upper(F.trim(F.col("fromStnCode").cast("string"))).alias("from_station_code"),
        F.upper(F.trim(F.col("toStnCode").cast("string"))).alias("to_station_code"),
        F.upper(F.trim(F.col("classCode").cast("string"))).alias("class_code"),
        F.col("baseFare").cast("double").alias("base_fare"),
        F.col("reservationCharge").cast("double").alias("reservation_charge"),
        F.col("superfastCharge").cast("double").alias("superfast_charge"),
        F.col("fuelAmount").cast("double").alias("fuel_amount"),
        F.col("totalConcession").cast("double").alias("total_concession"),
        F.col("tatkalFare").cast("double").alias("tatkal_fare"),
        F.col("serviceTax").cast("double").alias("service_tax"),
        F.col("otherCharge").cast("double").alias("other_charge"),
        F.col("cateringCharge").cast("double").alias("catering_charge"),
        F.col("dynamicFare").cast("double").alias("dynamic_fare"),
        F.col("totalFare").cast("double").alias("total_fare"),
        F.col("availability").cast("string").alias("availability"),
        F.col("distance").cast("double").alias("distance"),
        F.col("duration").cast("string").alias("duration"),
        F.col("timeStamp").cast("string").alias("timestamp")
    )
    .dropDuplicates()
)

display(prices.limit(10))


In [0]:
# Save raw cleaned Delta tables

schedules.write.mode("overwrite").format("delta").saveAsTable(f"{CATALOG}.{SCHEMA}.raw_schedules")
routes.write.mode("overwrite").format("delta").saveAsTable(f"{CATALOG}.{SCHEMA}.raw_train_routes")
prices.write.mode("overwrite").format("delta").saveAsTable(f"{CATALOG}.{SCHEMA}.raw_price_data")

print("Saved Delta tables:")
print(f"{CATALOG}.{SCHEMA}.raw_schedules")
print(f"{CATALOG}.{SCHEMA}.raw_train_routes")
print(f"{CATALOG}.{SCHEMA}.raw_price_data")
